CELDA 1: INSTALACION DEPENDENCIAS


In [1]:
%pip install gpxpy requests tqdm staticmap matplotlib opencv-python pillow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\echav\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Celda 2: Configuración y Imports

In [2]:
import os
import math
import json
import requests
import gpxpy
import cv2
import numpy as np
import matplotlib.pyplot as plt
from staticmap import StaticMap, CircleMarker, Line
from tqdm import tqdm
from PIL import Image
from io import BytesIO

CONFIG = {
    "gpx_file": "./DATA/2_beiras_final_track.gpx",
    "km_start": 137,
    "km_end": 147,
    "sampling_m": 15,  # Mayor distancia (Wikimedia tiene menos densidad)
    "output_dir": "./frames_wikimedia",
    "video_out": "wikimedia_video.mp4",
    "search_radius": 200  # Metros a buscar alrededor de cada punto
}

Celda 3: Utilidades y Parser

In [3]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    f1, f2, df, dl = map(math.radians, [lat1, lat2, lat2-lat1, lon2-lon1])
    a = math.sin(df/2)**2 + math.cos(f1)*math.cos(f2)*math.sin(dl/2)**2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1-a))

def parse_gpx(filepath):
    points = []
    with open(filepath) as f:
        for track in gpxpy.parse(f).tracks:
            for seg in track.segments:
                for p in seg.points:
                    points.append((p.latitude, p.longitude, p.elevation or 0))
    return points

def process_route(filepath, km_start, km_end, interval):
    raw_points = parse_gpx(filepath)
    filtered = []
    current_dist = 0
    acc_dist = 0
    last_pt = raw_points[0]
    start_km_dist = km_start * 1000
    
    for i, pt in enumerate(raw_points):
        d = haversine(last_pt[0], last_pt[1], pt[0], pt[1])
        current_dist += d
        
        if current_dist >= start_km_dist and len(filtered) == 0:
            filtered.append({'lat': pt[0], 'lon': pt[1], 'ele': pt[2], 'km': km_start})
            acc_dist = 0
            last_pt = pt
            continue
            
        if len(filtered) > 0:
            d = haversine(last_pt[0], last_pt[1], pt[0], pt[1])
            acc_dist += d
            if acc_dist >= interval:
                total_km = (current_dist / 1000)
                filtered.append({'lat': pt[0], 'lon': pt[1], 'ele': pt[2], 'km': total_km})
                acc_dist = 0
                last_pt = pt
        
        if km_end and (current_dist/1000) > km_end:
            break
    
    print(f"✅ {len(filtered)} puntos para procesar")
    return filtered

Celda 4: Descarga Mapillary y Gráficos

In [4]:
def _safe_json_response(response):
    """Devuelve dict/list JSON o None cuando la respuesta no es JSON válido."""
    if response is None:
        return None
    if response.status_code != 200:
        return None
    
    # Intento directo
    try:
        return response.json()
    except ValueError:
        pass
    
    # Fallback: algunos servicios devuelven prefijos o texto extra
    text = (response.text or "").strip()
    if not text:
        return None
    
    for marker in ("{", "["):
        start = text.find(marker)
        if start != -1:
            try:
                return json.loads(text[start:])
            except json.JSONDecodeError:
                continue
    return None

def get_wikimedia_images(lat, lon, radius=200, limit=1):
    """
    Busca imágenes geolocalizadas en Wikimedia Commons
    """
    url = "https://commons.wikimedia.org/w/api.php"
    
    params = {
        "action": "query",
        "list": "geosearch",
        "gscoord": f"{lat}|{lon}",
        "gsradius": radius,
        "gslimit": limit * 3,  # Pedir más para filtrar después
        "format": "json"
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        data = _safe_json_response(response)
        if not isinstance(data, dict):
            return None

        results = data.get("query", {}).get("geosearch", [])
        if not results:
            return None
        
        # Filtrar solo imágenes (no artículos)
        images = []
        for item in results:
            title = item.get("title", "")
            if ":" in title:  # Es un archivo
                ns = item.get("ns", 0)
                if ns == 6:  # Namespace de archivos
                    images.append({
                        "title": title,
                        "lat": item.get("lat"),
                        "lon": item.get("lon"),
                        "dist": item.get("dist")
                    })
        
        if images:
            images.sort(key=lambda x: x.get("dist", 9999))
            return images[0]
        return None
        
    except requests.RequestException:
        return None
    except Exception as e:
        print(f"⚠️ Error Wikimedia inesperado: {e}")
        return None

def get_image_url(title):
    """
    Obtiene la URL de descarga de una imagen de Wikimedia
    """
    url = "https://commons.wikimedia.org/w/api.php"
    params = {
        "action": "query",
        "titles": title,
        "prop": "imageinfo",
        "iiprop": "url",
        "format": "json"
    }
    
    try:
        r = requests.get(url, params=params, timeout=10)
        data = _safe_json_response(r)
        if not isinstance(data, dict):
            return None

        pages = data.get("query", {}).get("pages", {})
        for page in pages.values():
            img_info = page.get("imageinfo", [])
            if img_info:
                return img_info[0].get("url")
    except requests.RequestException:
        return None
    except Exception:
        return None
    
    return None

def download_wikimedia_image(url, filepath, size=(640, 640)):
    """Descarga y redimensiona imagen"""
    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 200:
            img = Image.open(BytesIO(response.content))
            img = img.convert('RGB')
            img = img.resize(size, Image.Resampling.LANCZOS)
            img.save(filepath, 'JPEG', quality=85)
            return True
    except Exception as e:
        print(f"⚠️ Error descargando: {e}")
    return False

def create_placeholder(filepath, text="Sin imagen"):
    """Crea frame negro con texto"""
    img = Image.new('RGB', (640, 640), color=(20, 20, 20))
    from PIL import ImageDraw, ImageFont
    
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 20)
    except:
        font = ImageFont.load_default()
    
    bbox = draw.textbbox((0, 0), text, font=font)
    tw = bbox[2] - bbox[0]
    th = bbox[3] - bbox[1]
    x = (640 - tw) // 2
    y = (640 - th) // 2
    
    draw.text((x, y), text, fill=(150, 150, 150), font=font)
    img.save(filepath, 'JPEG')

def process_wikimedia_route(points, output_dir, radius=200):
    """
    Procesa toda la ruta buscando imágenes en Wikimedia
    """
    os.makedirs(output_dir, exist_ok=True)
    stats = {"success": 0, "fallback": 0, "errors": 0}
    
    print("📥 Descargando imágenes de Wikimedia Commons...")
    
    for i, pt in enumerate(tqdm(points, desc="Procesando")):
        filepath = f"{output_dir}/frame_{i:04d}.jpg"
        
        img_data = get_wikimedia_images(pt['lat'], pt['lon'], radius=radius, limit=1)
        
        if img_data:
            img_url = get_image_url(img_data['title'])
            if img_url and download_wikimedia_image(img_url, filepath):
                stats["success"] += 1
                continue
        
        create_placeholder(filepath, f"KM {pt['km']:.1f}")
        stats["fallback"] += 1
    
    print(f"\n📊 Estadísticas:")
    print(f"   ✅ Imágenes encontradas: {stats['success']}")
    print(f"   🔄 Placeholders: {stats['fallback']}")
    print(f"   ❌ Errores: {stats['errors']}")
    
    return stats

Celda 5: Renderizado del Video

In [5]:
def generate_overlays(points, output_dir):
    """Genera mapa OSM y gráfica de elevación"""
    
    # 1. Mapa con StaticMap
    print("🗺️ Generando mapa...")
    m = StaticMap(800, 600)
    coords = [(p['lon'], p['lat']) for p in points]
    m.add_line(Line(coords, '#e11d48', 4))
    m.add_marker(CircleMarker((coords[0][0], coords[0][1]), '#22c55e', 10))
    m.add_marker(CircleMarker((coords[-1][0], coords[-1][1]), '#ef4444', 10))
    map_img = m.render()
    map_img.save(f"{output_dir}/map.png")
    
    # 2. Gráfica de elevación
    print("📈 Generando gráfica...")
    plt.figure(figsize=(10, 3), dpi=100)
    kms = [p['km'] for p in points]
    eles = [p['ele'] for p in points]
    
    plt.fill_between(kms, eles, color='#3b82f6', alpha=0.4)
    plt.plot(kms, eles, color='#3b82f6', linewidth=2)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.ylabel('Elevación (m)', fontsize=10)
    plt.xlabel('Kilómetros', fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/elevation.png", transparent=True, dpi=150)
    plt.close()
    
    print("✅ Overlays generados")

Celda 6: Ejecutar Todo

In [6]:
def create_final_video(output_dir, video_out, fps=20):
    """Une frames con overlays"""
    
    frames = sorted([f for f in os.listdir(output_dir) if f.startswith('frame_') and f.endswith('.jpg')])
    
    if not frames:
        print("❌ No hay frames para crear el video")
        return
    
    # Cargar overlays
    map_img = cv2.imread(f"{output_dir}/map.png")
    elev_img = cv2.imread(f"{output_dir}/elevation.png", cv2.IMREAD_UNCHANGED)
    
    if map_img is None or elev_img is None:
        print("⚠️ No se encontraron los overlays")
        return
    
    # Redimensionar overlays
    map_res = cv2.resize(map_img, (240, 180))
    elev_res = cv2.resize(elev_img, (500, 150))
    
    # Configurar video
    sample_frame = cv2.imread(f"{output_dir}/{frames[0]}")
    h, w, _ = sample_frame.shape
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(video_out, fourcc, fps, (w, h))
    
    print("🎬 Creando video...")
    
    for frame_name in tqdm(frames):
        frame = cv2.imread(f"{output_dir}/{frame_name}")
        
        # Pegar mapa (esquina superior izquierda)
        frame[10:10+map_res.shape[0], 10:10+map_res.shape[1]] = map_res
        
        # Pegar elevación (superior centro)
        x_start = (w - elev_res.shape[1]) // 2
        y_start = 10
        
        # Fondo negro para mejor legibilidad
        frame[y_start:y_start+elev_res.shape[0], x_start:x_start+elev_res.shape[1]] = 0
        frame[y_start:y_start+elev_res.shape[0], x_start:x_start+elev_res.shape[1]] = elev_res[:,:,:3]
        
        out.write(frame)
    
    out.release()
    print(f"✅ Video guardado: {video_out}")
    # 1. Procesar ruta
points = process_route(CONFIG['gpx_file'], CONFIG['km_start'], CONFIG['km_end'], CONFIG['sampling_m'])

# 2. Descargar imágenes de Wikimedia
process_wikimedia_route(points, CONFIG['output_dir'], radius=CONFIG['search_radius'])

# 3. Generar overlays
generate_overlays(points, CONFIG['output_dir'])

# 4. Crear video final
create_final_video(CONFIG['output_dir'], CONFIG['video_out'], fps=20)

✅ 325 puntos para procesar
📥 Descargando imágenes de Wikimedia Commons...


Procesando:   4%|▍         | 14/325 [00:05<01:57,  2.65it/s]


KeyboardInterrupt: 